# 02 — Cleaning & Feature Engineering

Reads `master_long.parquet`, cleans the signal, detects anomalies, and writes:

| Output | Description |
|--------|-------------|
| `master_long_clean.parquet` | Granular 1 Hz, cleaned, with anomaly flags |
| `master_short.parquet` | One row per participant × day, aggregated features |

In [ ]:
import sys, logging
from pathlib import Path

import pandas as pd
import numpy as np
import yaml

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.feature_engineering import (
    remove_hr_outliers,
    detect_time_gaps,
    interpolate_short_gaps,
    add_anomaly_flags,
    build_master_short,
)

PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
CONFIG_PATH   = REPO_ROOT / 'config' / 'participants.yaml'

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
logger = logging.getLogger('02_cleaning')

In [ ]:
# ── Load master_long ──────────────────────────────────────────────────────────
long_path = PROCESSED_DIR / 'master_long.parquet'

if not long_path.exists():
    raise FileNotFoundError(
        f'{long_path} not found.\n'
        'Run notebook 01_data_ingestion.ipynb first.'
    )

df = pd.read_parquet(long_path, engine='pyarrow')
print(f'Loaded master_long: {df.shape}')
df.head(3)

In [ ]:
# ── Step 1 — Remove physiologically impossible HR ─────────────────────────────
n_before = df['HR_bpm'].notna().sum()
df = remove_hr_outliers(df)
n_after  = df['HR_bpm'].notna().sum()
removed = n_before - n_after
print(f'Outlier HR removed : {removed:,} rows ({100*removed/len(df):.2f}%)')

In [ ]:
# ── Step 2 — Detect and mark time gaps ───────────────────────────────────────
df = df.groupby(['User_ID', 'Date'], group_keys=False).apply(detect_time_gaps)
n_gaps = df['gap_before'].sum()
print(f'Gap events detected: {n_gaps:,}')

In [ ]:
# ── Step 3 — Interpolate short gaps (≤ 5 s) ──────────────────────────────────
nan_before = df['HR_bpm'].isna().sum()
df = df.groupby(['User_ID', 'Date'], group_keys=False).apply(interpolate_short_gaps)
nan_after  = df['HR_bpm'].isna().sum()
filled = nan_before - nan_after
print(f'NaN rows filled by interpolation: {filled:,}')
print(f'Remaining NaN HR: {nan_after:,}')

In [ ]:
# ── Step 4 — Anomaly detection ────────────────────────────────────────────────
df = add_anomaly_flags(df, z_thresh=3.0, use_iqr=True)

n_anom = df['is_anomaly'].sum()
print(f'Anomalous rows (z-score OR IQR): {n_anom:,}  ({100*n_anom/len(df):.2f}%)')

In [ ]:
# ── Save master_long_clean ────────────────────────────────────────────────────
clean_path = PROCESSED_DIR / 'master_long_clean.parquet'
df.to_parquet(clean_path, index=False, engine='pyarrow')
print(f'Saved → {clean_path.relative_to(REPO_ROOT)}')
print(f'Shape  : {df.shape}')
df.dtypes

In [ ]:
# ── Load participant demographics from config ─────────────────────────────────
with open(CONFIG_PATH, encoding='utf-8') as fh:
    config = yaml.safe_load(fh)

demo_rows = []
for p in config.get('participants', []):
    demo_rows.append({
        'User_ID'   : p['folder'],
        'p_age'     : p.get('age'),
        'p_sex'     : p.get('sex'),
        'p_height'  : p.get('height_cm'),
        'p_weight'  : p.get('weight_kg'),
        'p_hr_max'  : p.get('hr_max'),
        'p_hr_rest' : p.get('hr_rest'),
        'p_vo2max'  : p.get('vo2max'),
    })

demo_df = pd.DataFrame(demo_rows)
print(demo_df)

In [ ]:
# ── Build and save master_short ───────────────────────────────────────────────
short = build_master_short(df, meta_df=demo_df)

short_path = PROCESSED_DIR / 'master_short.parquet'
short.to_parquet(short_path, index=False, engine='pyarrow')
print(f'Saved → {short_path.relative_to(REPO_ROOT)}')
print(f'Shape  : {short.shape}')
short.head()

In [ ]:
# ── Quality summary ───────────────────────────────────────────────────────────
print('=== Data quality per participant ===')
quality = (
    df.groupby('User_ID')
    .apply(lambda g: pd.Series({
        'total_rows'    : len(g),
        'valid_hr_pct'  : round(100 * g['HR_bpm'].notna().mean(), 2),
        'anomaly_pct'   : round(100 * g['is_anomaly'].mean(), 2),
        'sessions'      : g['Date'].nunique(),
    }))
    .reset_index()
)
print(quality.to_string(index=False))